In [ ]:
%pip install -q sentence-transformers
import os, subprocess, json
import numpy as np, pandas as pd
from google.colab import drive
drive.mount('/content/drive')
PASTA = "/content/drive/MyDrive/fakenews"

if not os.path.exists("FACTCK.BR"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jghm-f/FACTCK.BR.git"])
fc = pd.read_csv("FACTCK.BR/FACTCKBR.tsv", sep="\t")
print(fc.shape)   # esperado: (1313, 9)

Mounted at /content/drive
(1313, 9)


In [ ]:
# Indexamento

from sentence_transformers import SentenceTransformer
emb = SentenceTransformer("intfloat/multilingual-e5-small")

# AJUSTE os nomes das colunas conforme fc.columns
COL_ALEGACAO, COL_LINK, COL_VEREDITO = "claimReviewed", "URL", "alternativeName"

fc = fc.dropna(subset=[COL_ALEGACAO]).reset_index(drop=True)
E = emb.encode(["passage: " + str(x) for x in fc[COL_ALEGACAO]],
               normalize_embeddings=True, batch_size=64, show_progress_bar=True)
print(E.shape)

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

model.safetensors: downloading bytes:           |  0.00B            

... (6 outputs left out)

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Batches:   0%|          | 0/21 [00:00<?, ?it/s]

(1300, 384)


In [ ]:
# Função de Busca

def buscar(texto, k=3, limiar=0.80):
    q = emb.encode(["query: " + texto[:500]], normalize_embeddings=True)[0]
    sims = E @ q                                   # cosseno (vetores normalizados)
    idx = np.argsort(-sims)[:k]
    return [{"alegacao": fc.loc[i, COL_ALEGACAO], "similaridade": round(float(sims[i]), 3),
             "veredito_checagem": str(fc.loc[i, COL_VEREDITO]), "link": str(fc.loc[i, COL_LINK])}
            for i in idx if sims[i] >= limiar]

In [ ]:
amostra = fc.sample(10, random_state=3)
for i, r in amostra.iterrows():
    print(i, "|", r["claimReviewed"][:140])

583 | Nós deixamos prescrever crime de júri, homicídio e tentativa de homicídio dolosos, prescrição de 20 anos. Isso tudo na primeira instância.
871 | Por pressão da bancada evangélica, Lei Maria da Penha será revogada pela ministra Damares
119 | Vai vendo!!! Times Square agora há pouco!!! A esquerda toda se coçando de raiva!!! Hehehe!!! Vai nessa, Capitão!!!
1115 | Principal aliado de Bolsonaro no Amapá acaba de ser preso pela PF por diversos crimes, ele pedia o fim da corrupção e a prisão de Lula
1190 | FHC: 'Não há saída: entre Haddad e Bolsonaro, vou de Haddad'
999 | Na Suíça, possuir armas é obrigatório
144 | Que bonito, ein, PT Quando foi que essa senhora [Aleida March] trabalhou no Brasil A profissão dela era ser amiguinha do Lula
1158 | Aposentadoria: José Sarney, R$ 109.982. Temer, R$ 68.985. FHC, 67.450. Joaquim Barbosa, R$ 46.320. Eliseu Padilha: R$ 32.467. Alvaro Dias, R
293 | FUNCIONRIA DO IBOPE E DATA FOLHA DESMASCARA O INSTITUTO E FALA A VERDADE
440 | Mais de 100 milhões

In [ ]:
# PREENCHA: índice da alegação (do print acima) : sua paráfrase
casam = {
    583:  "Dizem que deixaram prescrever, depois de 20 anos e ainda na primeira instância, os crimes de homicídio doloso e tentativa que vão a júri.",
    871:  "Circula que a ministra Damares vai revogar a Lei Maria da Penha porque a bancada evangélica pressionou.",
    119:  "Imagem da Times Square, em Nova York, com uma mensagem de apoio ao Bolsonaro, e a esquerda ficou furiosa.",
    1115: "Um dos maiores aliados do presidente no Amapá foi preso pela Polícia Federal por vários crimes, embora defendesse o fim da corrupção e a cadeia para o Lula.",
    1190: "Fernando Henrique Cardoso teria dito que, entre Haddad e Bolsonaro, o voto dele vai para o Haddad.",
    999:  "Dizem que na Suíça todo cidadão é obrigado por lei a ter uma arma em casa.",
    144:  "Postagem ironiza a Aleida March, dizendo que ela nunca trabalhou no Brasil e que sua única ocupação era ser amiga do Lula.",
    1158: "Circula uma lista com aposentadorias de políticos e ex-presidentes como Sarney, Temer e FHC, todas acima de 30 mil reais por mês.",
    293:  "Uma funcionária do Ibope e do Datafolha teria exposto o instituto e revelado a verdade sobre as pesquisas eleitorais.",
    440:  "Mais de cem milhões de brasileiros não têm rede de esgoto e jogam os dejetos em fossas ou valas a céu aberto.",
}

nao_casam = [
    # sem relação nenhuma
    "Receita de bolo de cenoura com cobertura de chocolate.",
    "O time venceu o jogo de ontem por dois a zero.",
    "A previsão do tempo para o fim de semana indica chuva no litoral.",
    "Como configurar o roteador para melhorar o sinal do wifi em casa.",
    "A biblioteca municipal muda o horário de funcionamento durante as férias.",
    # difíceis: assunto parecido, mas não é nenhuma das alegações
    "A Câmara discute mudanças no Código Penal, mas ainda não há data para votação.",
    "A ministra participou de um evento sobre políticas públicas para mulheres em Brasília.",
    "O Tribunal Superior Eleitoral divulgou o calendário oficial das próximas eleições.",
    "O governo anunciou investimentos em obras de saneamento básico na região Nordeste.",
    "A Suíça tem um dos maiores índices de qualidade de vida da Europa.",
]

def scores(texto):
    q = emb.encode(["query: " + texto], normalize_embeddings=True)[0]
    return E @ q

# positivos: a checagem certa aparece no top-k? com que score?
rows = []
for idx, t in casam.items():
    s = scores(t)
    ordem = np.argsort(-s)
    rows.append({"idx": idx, "score_certa": round(float(s[idx]), 3),
                 "posicao": int(np.where(ordem == idx)[0][0]) + 1,
                 "top1_score": round(float(s.max()), 3)})
pos = pd.DataFrame(rows)
print(pos)
print("top-1:", (pos.posicao == 1).mean(), "| top-3:", (pos.posicao <= 3).mean())

# negativos: maior score que qualquer checagem recebeu
neg = [float(scores(t).max()) for t in nao_casam]
print("\nnegativos (maior score):", [round(x, 3) for x in neg])

# varredura de limiar
print("\nlimiar | acha a certa (positivos) | falso casamento (negativos)")
for lim in np.arange(0.70, 0.95, 0.02):
    acha = (pos.score_certa >= lim).mean()
    falso = np.mean([x >= lim for x in neg])
    print(f"{lim:.2f}   | {acha:.0%}                    | {falso:.0%}")

    idx  score_certa  posicao  top1_score
0   583        0.916        1       0.916
1   871        0.967        1       0.967
2   119        0.877        6       0.886
3  1115        0.907        1       0.907
4  1190        0.892        1       0.892
5   999        0.910        1       0.910
6   144        0.884        1       0.884
7  1158        0.863       29       0.880
8   293        0.912        1       0.912
9   440        0.892        1       0.892
top-1: 0.8 | top-3: 0.8

negativos (maior score): [0.864, 0.874, 0.865, 0.863, 0.873, 0.876, 0.873, 0.873, 0.883, 0.865]

limiar | acha a certa (positivos) | falso casamento (negativos)
0.70   | 100%                    | 100%
0.72   | 100%                    | 100%
0.74   | 100%                    | 100%
0.76   | 100%                    | 100%
0.78   | 100%                    | 100%
0.80   | 100%                    | 100%
0.82   | 100%                    | 100%
0.84   | 100%                    | 100%
0.86   | 100%                   

In [ ]:
amostra = fc.sample(15, random_state=7)
for i, r in amostra.iterrows():
    print(i, "|", r["claimReviewed"][:200])

347 | PF descobre ilha de R$ 4 milhões da família Lula
1160 | O medicamento exposto abaixo, Dipirona S-500, importado da Venezuela, contém um vírus perigosíssimo, não tomem... !
374 | Paraná Pesquisas registrado no TSE: O mito Bolsonaro vence em todos os estados, porque o povo quer renovação!
361 | Vídeo diz que eleição de 2014 foi fraudada
1215 | Jesus é travesti imagem
500 | Enquanto o Brasil acompanha a Copa, os deputados tentarão criar 400 novos municípios.
1284 | General Villas Bôas anuncia intervenção militar e convoca povo do Brasil
790 | Pensão por morte. Aposentadoria atual: 100% do salário; Reforma da Previdência: 50% do salário, +10% por dependente
34 | Enquanto trabalhadores seguem para trabalhar, alguns grevistas pararam para telefonar
825 | General Augusto Heleno pede a prisão do jornalista da Record Paulo Henrique Amorim, por ameaça de morte ao presidente Bolsonaro
956 | Câmera em carro flagra momento exato do acidente com o 737 MAX 8 da Boeing na Etiópia
830 | Cuidado p

In [ ]:
pos_llm, neg_llm = pos.copy(), list(neg)

Segunda rodada da calibração do limiar: 30 alegações sorteadas, cada uma com uma paráfrase (rascunhada por LLM, sem edição), e 15 frases negativas.

In [ ]:
amostra = fc.sample(30, random_state=7)
for i, r in amostra.iterrows():
    print(i, "|", r["claimReviewed"][:200])

347 | PF descobre ilha de R$ 4 milhões da família Lula
1160 | O medicamento exposto abaixo, Dipirona S-500, importado da Venezuela, contém um vírus perigosíssimo, não tomem... !
374 | Paraná Pesquisas registrado no TSE: O mito Bolsonaro vence em todos os estados, porque o povo quer renovação!
361 | Vídeo diz que eleição de 2014 foi fraudada
1215 | Jesus é travesti imagem
500 | Enquanto o Brasil acompanha a Copa, os deputados tentarão criar 400 novos municípios.
1284 | General Villas Bôas anuncia intervenção militar e convoca povo do Brasil
790 | Pensão por morte. Aposentadoria atual: 100% do salário; Reforma da Previdência: 50% do salário, +10% por dependente
34 | Enquanto trabalhadores seguem para trabalhar, alguns grevistas pararam para telefonar
825 | General Augusto Heleno pede a prisão do jornalista da Record Paulo Henrique Amorim, por ameaça de morte ao presidente Bolsonaro
956 | Câmera em carro flagra momento exato do acidente com o 737 MAX 8 da Boeing na Etiópia
830 | Cuidado p

In [ ]:
# PREENCHA: índice da alegação (do print acima) : sua paráfrase
casam = {
    347:  "A Polícia Federal teria encontrado uma ilha de quatro milhões de reais pertencente à família do Lula.",
    1160: "Circula um alerta dizendo que a dipirona S-500 trazida da Venezuela vem contaminada com um vírus perigoso e que ninguém deve tomar.",
    374:  "Uma pesquisa do Paraná Pesquisas, supostamente registrada no TSE, mostraria o Bolsonaro vencendo em todos os estados.",
    361:  "Um vídeo afirma que houve fraude na eleição presidencial de 2014.",
    1215: "Circula uma imagem que mostraria Jesus como travesti.",                                   # MUDAR
    500:  "Enquanto todo mundo assiste à Copa, os deputados estariam articulando a criação de 400 novos municípios.",
    1284: "Mensagem diz que o general Villas Bôas anunciou uma intervenção militar e chamou a população para participar.",
    790:  "Comparação diz que a pensão por morte, hoje de 100% do salário, cairia para 50% mais 10% por dependente com a reforma da Previdência.",
    34:   "Imagem mostra grevistas parados no celular enquanto os outros trabalhadores seguem para o serviço.",   # MUDAR
    825:  "Dizem que o general Augusto Heleno pediu a prisão do jornalista Paulo Henrique Amorim, da Record, por ameaça de morte ao presidente Bolsonaro.",
    956:  "Vídeo gravado por uma câmera dentro de um carro teria flagrado o instante exato da queda do Boeing 737 MAX 8 na Etiópia.",
    830:  "Alerta diz para não tomar o paracetamol com a marca P-500, que médicos teriam provado conter o vírus Machupo.",   # MUDAR
    618:  "Segundo um vazamento do WikiLeaks, a Globo teria repassado à Unesco só 10% do dinheiro arrecadado no Criança Esperança.",
    1159: "Apreenderam seis mil toras de mogno, no valor de 8 milhões de reais, e o dono seria o marido da Marina, apontado como um dos maiores desmatadores da Amazônia.",
    528:  "Afirma-se que Pedro Parente estava no Banco Central durante a hiperinflação do governo Sarney.",   # MUDAR
    1256: "Mensagem de campanha da Marina Silva, candidata número 17, pedindo que o país não desista.",   # MUDAR
    164:  "Dizem que a mãe e a irmã do Adélio trabalhavam no gabinete da deputada Maria do Rosário, e que a imprensa de esquerda não comenta isso.",
    51:   "Balanço dos protestos do dia diria que só 1% defendia a educação e que 90% estavam ali por Lula livre.",   # MUDAR
    534:  "Um político afirma que terminou seus mandatos com 87% de aprovação popular.",   # MUDAR
    237:  "Alerta aos eleitores: tirar foto do comprovante de votação seria crime eleitoral.",
    731:  "Imagens da internet mostrariam a Funai, com apoio da Força Nacional armada, entrando em fazendas para colocar indígenas dentro delas.",   # MUDAR
    152:  "Mensagem urgente diz que a China mandou um recado aos EUA e ao mundo para se prepararem para uma Terceira Guerra Mundial.",
    113:  "Notícia diz que o Exército perfurou um poço em Caicó, no Nordeste, e achou água jorrando num terreno onde hoje funciona um complexo de creches.",   # MUDAR
    1063: "Circula que o filho do Lula tem uma fazenda avaliada em 50 milhões de reais.",
    725:  "Alguém afirma que as drogas causam 90% dos crimes violentos do país.",   # MUDAR
    253:  "Dizem que o Ciro Gomes voltou da Europa e declarou voto no Bolsonaro.",
    598:  "Compara-se que os tiroteios em massa mataram cerca de 300 pessoas nos EUA em 2017, enquanto no Brasil morrem em média 120 pessoas por dia baleadas.",   # MUDAR
    928:  "Alguém teria transferido os chefes das maiores facções criminosas para presídios de segurança máxima.",   # MUDAR
    351:  "O papa teria mandado um terço para o Lula, que estaria preso há 67 dias.",
    804:  "Em meio a ataques na Câmara, o Moro teria dito que, se falasse a verdade, teria de prender todo mundo e fechar o Congresso.",
}

nao_casam = [
    # sem relação nenhuma
    "Receita de pão de queijo com polvilho azedo.",
    "O time venceu a partida de ontem por três a um.",
    "A previsão do tempo indica sol e temperatura alta durante a semana.",
    "Como trocar o pneu do carro em poucos passos.",
    "A biblioteca da universidade amplia o horário de funcionamento nas provas.",
    # difíceis: mesmo assunto de alguma alegação, mas não é nenhuma delas
    "A Polícia Federal fez uma operação nesta manhã contra fraudes em licitações em três estados.",
    "O TSE lançou um aplicativo para o eleitor consultar o local de votação.",
    "A Anvisa publicou uma nota explicando como descartar medicamentos vencidos.",
    "A Câmara aprovou um projeto que muda as regras de repasse de verbas aos municípios.",
    "O governo enviou ao Congresso o texto de uma reforma da Previdência, que ainda será debatido.",
    "O Inpe divulgou os dados mensais de desmatamento na Amazônia.",
    "A Boeing informou que está revisando os procedimentos de segurança de seus aviões.",
    "O Exército distribui água em carros-pipa para cidades do semiárido nordestino.",
    "Representantes da Funai participaram de uma audiência pública sobre demarcação de terras.",
    "A Unesco divulgou um relatório sobre a educação no Brasil.",
]

def scores(texto):
    q = emb.encode(["query: " + texto], normalize_embeddings=True)[0]
    return E @ q

# positivos: a checagem certa aparece no top-k? com que score?
rows = []
for idx, t in casam.items():
    s = scores(t)
    ordem = np.argsort(-s)
    rows.append({"idx": idx, "score_certa": round(float(s[idx]), 3),
                 "posicao": int(np.where(ordem == idx)[0][0]) + 1,
                 "top1_score": round(float(s.max()), 3)})
pos = pd.DataFrame(rows)
print(pos)
print("top-1:", (pos.posicao == 1).mean(), "| top-3:", (pos.posicao <= 3).mean())

# negativos: maior score que qualquer checagem recebeu
neg = [float(scores(t).max()) for t in nao_casam]
print("\nnegativos (maior score):", [round(x, 3) for x in neg])

# varredura de limiar
print("\nlimiar | acha a certa (positivos) | falso casamento (negativos)")
for lim in np.arange(0.70, 0.95, 0.02):
    acha = (pos.score_certa >= lim).mean()
    falso = np.mean([x >= lim for x in neg])
    print(f"{lim:.2f}   | {acha:.0%}                    | {falso:.0%}")

     idx  score_certa  posicao  top1_score
0    347        0.924        1       0.924
1   1160        0.920        1       0.920
2    374        0.913        2       0.930
3    361        0.936        1       0.936
4   1215        0.928        1       0.928
5    500        0.934        1       0.934
6   1284        0.920        1       0.920
7    790        0.884        1       0.884
8     34        0.920        1       0.920
9    825        0.941        1       0.941
10   956        0.944        1       0.944
11   830        0.939        1       0.939
12   618        0.937        2       0.942
13  1159        0.920        1       0.920
14   528        0.953        1       0.953
15  1256        0.897        1       0.897
16   164        0.925        1       0.925
17    51        0.884        2       0.893
18   534        0.929        1       0.929
19   237        0.919        1       0.919
20   731        0.937        1       0.937
21   152        0.929        1       0.929
22   113   